# Module 06: Error Handling & Logging — Solutions

Complete solutions for all exercises.

## Exercise 1: Safe Division

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print('Error: Cannot divide by zero')
        return None
    except TypeError:
        print('Error: Both inputs must be numbers')
        return None
    else:
        return result
    finally:
        print('Operation attempted')

print('Test 1:', safe_divide(10, 2))
print('Test 2:', safe_divide(10, 0))
print('Test 3:', safe_divide(10, 'a'))

## Exercise 2: Custom Exception for Data Validation

In [ ]:
class DataQualityError(Exception):
    pass

def check_data_quality(data):
    if len(data) == 0:
        raise DataQualityError('Data list is empty')
    
    for val in data:
        if not isinstance(val, (int, float)):
            raise DataQualityError('Value ' + str(val) + ' is not a number')
        if val < 0:
            raise DataQualityError('Negative value found: ' + str(val))
    
    return sum(data) / len(data)

# Tests
tests = [
    [1, 2, 3, 4, 5],
    [],
    [1, -2, 3],
    [1, 'bad', 3],
]

for i, data in enumerate(tests):
    print('Test', i + 1, ':', data)
    try:
        result = check_data_quality(data)
        print('  Mean:', result)
    except DataQualityError as e:
        print('  Error:', e)

## Exercise 3: Logging Setup

In [ ]:
import logging

# Create logger
logger = logging.getLogger('ml_pipeline')
logger.setLevel(logging.DEBUG)

# Console handler
console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))

# File handler
file_handler = logging.FileHandler('ml_pipeline.log', mode='w')
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(logging.Formatter('%(asctime)s - %(message)s'))

logger.addHandler(console)
logger.addHandler(file_handler)

# Log messages
logger.debug('Initializing pipeline')
logger.info('Loading dataset')
logger.warning('Dataset has 10 percent missing values')
logger.info('Training started')
logger.error('Training failed: NaN detected')

print('Check ml_pipeline.log for full debug output')

## Exercise 4: Robust File Loader

In [ ]:
import json

def load_json_safe(path):
    try:
        with open(path, 'r') as f:
            data = json.load(f)
        print('Loaded successfully')
        return data
    except FileNotFoundError:
        print('Warning: File not found, using defaults')
        return {'status': 'default'}
    except json.JSONDecodeError:
        print('Error: Invalid JSON in file')
        return {}
    finally:
        print('Load attempt completed')

# Test 1: Valid file
with open('valid.json', 'w') as f:
    json.dump({'key': 'value'}, f)
print('=== Test 1: Valid file ===')
print(load_json_safe('valid.json'))
print()

# Test 2: Missing file
print('=== Test 2: Missing file ===')
print(load_json_safe('missing.json'))
print()

# Test 3: Invalid JSON
with open('invalid.json', 'w') as f:
    f.write('{not json}')
print('=== Test 3: Invalid JSON ===')
print(load_json_safe('invalid.json'))

## Exercise 5: Training Loop with Error Handling

In [ ]:
import logging
import random

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s - %(message)s'
)
logger = logging.getLogger('training')

random.seed(42)
total_epochs = 10
failed = 0
successful = 0

for epoch in range(1, total_epochs + 1):
    try:
        if random.random() < 0.2:
            raise ValueError('Random training error')
        logger.info('Epoch %d/%d completed', epoch, total_epochs)
        successful += 1
    except ValueError as e:
        logger.error('Epoch %d failed: %s', epoch, e)
        failed += 1

print()
print('Summary: Total:', total_epochs, '| Successful:', successful, '| Failed:', failed)

## Exercise 6: Pipeline with Fallback

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s: %(message)s'
)
logger = logging.getLogger('fallback')

def load_dataset_or_fallback(paths):
    last_error = None
    for path in paths:
        try:
            logger.info('Attempting to load: %s', path)
            with open(path, 'r') as f:
                content = f.read()
            logger.info('Successfully loaded: %s', path)
            return content
        except FileNotFoundError as e:
            logger.warning('Failed to load: %s', path)
            last_error = e
    
    raise FileNotFoundError('No data source available')

# Create only the last file
with open('final_source.txt', 'w') as f:
    f.write('data from final source')

# Try paths where only the last one exists
paths = ['missing1.txt', 'missing2.txt', 'final_source.txt']
try:
    data = load_dataset_or_fallback(paths)
    print('Loaded data:', data)
except FileNotFoundError as e:
    print('Error:', e)